# Fine-tuning Qwen3.5-0.8B con LoRA (Unsloth) - FarmifAI

Este notebook entrena el modelo `unsloth/Qwen3.5-0.8B` para la tarea de RAG agrícola usando LoRA.
Se asegura la compatibilidad con T4 (fp16) y se realiza una evaluación completa.

## 1. Setup e Instalación

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -qqq rouge-score nltk sacrebleu scikit-learn sentence-transformers selfcheckgpt

In [ ]:
import torch
gpu_props = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_props.name}")
print(f"VRAM: {gpu_props.total_memory / 1024**3:.1f} GB")
print(f"bf16 support: {torch.cuda.is_bf16_supported()}")

if not torch.cuda.is_bf16_supported():
    print("\n[WARNING] bf16 NO soportado. Se usar\u00e1 fp16.")
    DTYPE = torch.float16
else:
    DTYPE = torch.bfloat16

## 2. Configuración de Hiperparámetros

In [ ]:
# --- Modelo ---
MODEL_NAME = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 4096

# --- LoRA Hyperparameters ---
LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0
USE_RSLORA = False
RANDOM_STATE = 3407

# --- Training Hyperparameters ---
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
WARMUP_STEPS = 30
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"
OPTIMIZER = "adamw_8bit"
LOGGING_STEPS = 1

# --- Dataset ---
EVAL_SPLIT_RATIO = 0.10
SEED = 42

# --- Exportación ---
GGUF_QUANT_METHOD = "q4_k_m"
HF_REPO_NAME = "FarmifAI/Qwen3.5-0.8B_FarmifAI3.0"
HF_TOKEN = ""  # Dejar en blanco, se pedirá más adelante si es necesario

# --- Evaluación ---
NUM_EVAL_SAMPLES = 50
SELFCHECK_NUM_SAMPLES = 5
MAX_NEW_TOKENS = 512

## 3. Carga del Modelo y LoRA Adapters

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = False,  # Usar LoRA 16-bit (no QLoRA)
    use_gradient_checkpointing = "unsloth",
)
# Obtener el tokenizador de texto si es un procesador multimodal
tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = RANDOM_STATE,
    use_rslora = USE_RSLORA,
    loftq_config = None,
)

## 4. Carga y Preparación del Dataset

In [ ]:
from google.colab import files
import json, random

print("[INFO] Sube tu archivo JSONL de entrenamiento:")
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"[OK] Archivo recibido: {dataset_filename}")

with open(dataset_filename, 'r', encoding='utf-8') as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

random.seed(SEED)
random.shuffle(raw_data)

eval_size = max(1, int(len(raw_data) * EVAL_SPLIT_RATIO))
train_data = raw_data[eval_size:]
eval_data = raw_data[:eval_size]

print(f"[INFO] Split: Entrenamiento {len(train_data)}, Evaluación {len(eval_data)}")

train_filename = dataset_filename.replace('.jsonl', '_train.jsonl')
eval_filename = dataset_filename.replace('.jsonl', '_eval.jsonl')

with open(train_filename, 'w', encoding='utf-8') as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

with open(eval_filename, 'w', encoding='utf-8') as f:
    for item in eval_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("[INFO] Descargando splits...")
files.download(train_filename)
files.download(eval_filename)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        for convo in convos
    ]
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    return tokenized

train_dataset = train_dataset.map(formatting_prompts_func, batched=True, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True, remove_columns=eval_dataset.column_names)
print("[OK] Dataset formateado")

## 5. Entrenamiento

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        remove_unused_columns=False,
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=LOGGING_STEPS,
        optim=OPTIMIZER,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_steps=WARMUP_STEPS,
        seed=RANDOM_STATE,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"\n[OK] Entrenamiento completado en {trainer_stats.metrics['train_runtime']/60:.2f} min.")
print(f"Peak VRAM = {used_memory} GB.")
print(f"VRAM usada por LoRA = {used_memory_for_lora} GB.")

## 6. Generación para Evaluación

In [ ]:
import re
import numpy as np
from collections import Counter
from transformers import TextStreamer

def check_format_adherence(text):
    reasoning_matches = re.findall(r'<reasoning>(.*?)</reasoning>', text, re.DOTALL)
    answer_matches = re.findall(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if len(reasoning_matches) == 1 and len(answer_matches) == 1:
        return text.find('<reasoning>') < text.find('<answer>')
    return False

def extract_answer(text):
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()

def extract_knowledge(user_content):
    match = re.search(r'<knowledge>(.*?)</knowledge>', user_content, re.DOTALL)
    return match.group(1).strip() if match else ""

def count_syllables_spanish(word):
    word = word.lower()
    vowels = 'aeiou\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc'
    count = sum(1 for char in word if char in vowels)
    return max(1, count)

def flesch_reading_ease_spanish(text):
    sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    words = re.findall(r'\w+', text)
    if not words or not sentences: return 0.0
    score = 206.835 - 62.3 * (sum(count_syllables_spanish(w) for w in words) / len(words)) - (len(words) / len(sentences))
    return round(score, 2)

In [ ]:
FastLanguageModel.for_inference(model)
eval_subset = eval_data[:min(NUM_EVAL_SAMPLES, len(eval_data))]
results = []
print(f"[INFO] Generando respuestas para {len(eval_subset)} muestras...")

for i, sample in enumerate(eval_subset):
    messages = sample["messages"]
    user_msg = next((m for m in messages if m["role"] == "user"), None)
    knowledge = extract_knowledge(user_msg["content"]) if user_msg else ""
    
    assistant_msg = next((m for m in messages if m["role"] == "assistant"), None)
    reference = assistant_msg["content"] if assistant_msg else ""
    reference_answer = extract_answer(reference)
    
    input_messages = [m for m in messages if m["role"] != "assistant"]
    input_text = tokenizer.apply_chat_template(input_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).to("cuda")
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, use_cache=True, temperature=0.7, top_p=0.9, do_sample=True)
    
    generated_text = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    generated_answer = extract_answer(generated_text)
    
    results.append({
        "index": i, "knowledge": knowledge,
        "reference_answer": reference_answer,
        "generated_full": generated_text,
        "generated_answer": generated_answer,
    })
print("[OK] Generaci\u00f3n completada")

## 7. Métricas de Evaluación (ROUGE, BLEU, NLI, SelfCheckGPT)

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder

format_scores = [check_format_adherence(r["generated_full"]) for r in results]
format_adherence = sum(format_scores) / len(format_scores) * 100 if format_scores else 0

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
rouge_results = {"rouge1": [], "rouge2": [], "rougeL": []}
for r in results:
    scores = scorer.score(r["reference_answer"], r["generated_answer"])
    for key in rouge_results: rouge_results[key].append(scores[key].fmeasure)

bleu = sacrebleu.corpus_bleu([r["generated_answer"] for r in results], [[r["reference_answer"] for r in results]])
readability_scores = [flesch_reading_ease_spanish(r["generated_answer"]) for r in results]

embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
ref_embeddings = embed_model.encode([r["reference_answer"] for r in results])
gen_embeddings = embed_model.encode([r["generated_answer"] for r in results])
cos_similarities = [cosine_similarity([ref], [gen])[0][0] for ref, gen in zip(ref_embeddings, gen_embeddings)]

nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')
nli_results = {"entailment": 0, "contradiction": 0, "neutral": 0}
for r in results:
    if r["knowledge"] and r["generated_answer"]:
        scores = nli_model.predict([(r["knowledge"], r["generated_answer"])])
        nli_results[['contradiction', 'entailment', 'neutral'][np.argmax(scores[0])]] += 1
total_nli = sum(nli_results.values())
nli_percentages = {k: v/total_nli*100 if total_nli else 0 for k, v in nli_results.items()}

del nli_model; torch.cuda.empty_cache()

In [ ]:
selfcheck_subset = results[:min(20, len(results))]
selfcheck_scores = []
print(f"[INFO] SelfCheckGPT: {SELFCHECK_NUM_SAMPLES} samples por muestra...")

for idx, r in enumerate(selfcheck_subset):
    input_messages = [m for m in eval_data[r["index"]]["messages"] if m["role"] != "assistant"]
    input_text = tokenizer.apply_chat_template(input_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).to("cuda")
    
    stoch_answers = []
    for _ in range(SELFCHECK_NUM_SAMPLES):
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=1.0, top_p=0.95, do_sample=True)
        stoch_answers.append(extract_answer(tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)))
    
    stoch_embs = embed_model.encode(stoch_answers)
    selfcheck_scores.append(np.mean([cosine_similarity(embed_model.encode([r["generated_answer"]]), [se])[0][0] for se in stoch_embs]))

avg_selfcheck = np.mean(selfcheck_scores) if selfcheck_scores else 0

In [ ]:
print("=" * 60)
print("[INFO] REPORTE DE EVALUACI\u00d3N")
print("=" * 60)
print(f"Format Adherence:      {format_adherence:.1f}%")
print(f"ROUGE-1 (F1):          {np.mean(rouge_results['rouge1']):.4f}")
print(f"BLEU:                  {bleu.score:.2f}")
print(f"Legibilidad (Flesch):  {np.mean(readability_scores):.1f}")
print(f"Cosine Similarity:     {np.mean(cos_similarities):.4f}")
print(f"NLI Entailment:        {nli_percentages['entailment']:.1f}%")
print(f"SelfCheck Consistency: {avg_selfcheck:.4f}")
print("=" * 60)

## 8. Exportar Modelo (GGUF y HuggingFace)

In [ ]:
model.save_pretrained("farmifai_lora")
tokenizer.save_pretrained("farmifai_lora")
print("[OK] LoRA adapter guardado localmente en 'farmifai_lora/'")

print(f"[INFO] Exportando a GGUF ({GGUF_QUANT_METHOD})...")
model.save_pretrained_gguf("farmifai_gguf", tokenizer, quantization_method=GGUF_QUANT_METHOD)

import glob
for f in glob.glob("farmifai_gguf/*.gguf"):
    print(f"[INFO] Descargando: {f}")
    files.download(f)

In [ ]:
from huggingface_hub import login
if not HF_TOKEN:
    HF_TOKEN = input("[INPUT] Ingresa tu HuggingFace Token: ")
login(token=HF_TOKEN)

print(f"[INFO] Subiendo a HuggingFace: {HF_REPO_NAME}")
model.push_to_hub_gguf(
    HF_REPO_NAME,
    tokenizer,
    quantization_method=GGUF_QUANT_METHOD,
    token=HF_TOKEN,
)
print("[OK] Subida exitosa.")